# Parts 1–3 — Problem Selection, Literature Review, and Dataset

This notebook establishes the research problem before any model comparison. It documents the five-paper review, the public datasets, the prediction/relevance target, data provenance, and the frozen development/test boundary.

**Notebook status:** executed assignment evidence. All quantitative analysis is development-only; the locked 60-question test partition is not read for evaluation.


## Part 1 — Selected real-world problem

**Problem:** retrieve trustworthy Urdu-script evidence when a user asks a noisy Roman-Urdu question.

Roman Urdu is written with Latin characters, has no standard spelling, and often includes English words. Most authoritative local content is in Urdu script. Direct word matching therefore misses relevant passages, while a semantic model can return something generally related but factually unresponsive. The practical user-facing task is not free-form text generation: given a query and a fixed Urdu corpus, rank passages so the correct evidence appears near the top, then return an exact supported sentence or abstain.

**Research question:** Can script-aware multi-query reformulation and character-level title matching improve top-10 evidence retrieval over direct dense retrieval, single transliteration, and a conventional lexical+dense hybrid?

**Machine-learning formulation:** Each query–passage pair has a binary relevance target. The diagnostic CSV stores the verified relevant passage identifier for each query; evaluation asks whether that identifier appears at ranks 1, 5, or 10. The interface adds a separate deterministic evidence-validity decision after retrieval.

This information-retrieval problem is outside the assignment's example list. The repository can demonstrate the technical work but cannot prove an administrative approval; the required prior approval should therefore be retained with the submission records.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Part 2 — Literature review protocol

Five directly relevant papers were selected to cover the complete chain: Roman-Urdu data, transliteration, Roman-Urdu retrieval, multilingual dense retrieval, and neural reranking. Four of the five were published from 2022–2025, satisfying the requirement that at least three be from the last five years.

The review compares each paper's approach, evaluation focus, and unresolved gap. The common challenges are spelling variation, cross-script mismatch, scarce relevance labels, entity confusion, and the CPU cost of reranking. Common metrics include character error rate for transliteration and Recall/MRR/nDCG for retrieval.


In [2]:
papers = [
    {"year": 2022, "paper": "Alam & Hussain — Roman-Urdu-Parl", "approach": "parallel Roman-Urdu/Urdu corpus", "evaluation": "corpus quality and alignment", "gap": "no passage-retrieval protocol"},
    {"year": 2025, "paper": "Butt et al. — Roman-Urdu transliteration", "approach": "transformer transliteration", "evaluation": "character-level transliteration quality", "gap": "one conversion can remain ambiguous for search"},
    {"year": 2025, "paper": "Butt et al. — Roman Urdu IR", "approach": "large IR dataset and multilingual baseline", "evaluation": "ranked retrieval metrics", "gap": "translated benchmark; limited route transparency"},
    {"year": 2024, "paper": "Wang et al. — Multilingual E5", "approach": "multilingual contrastive text embeddings", "evaluation": "multilingual retrieval benchmarks", "gap": "semantic similarity can confuse fine-grained entities"},
    {"year": 2019, "paper": "Nogueira & Cho — BERT reranking", "approach": "query–passage cross-encoder", "evaluation": "MRR and passage ranking", "gap": "high CPU cost and no evidence guarantee"},
]
print_table(papers, ["year", "paper", "approach", "evaluation", "gap"])
recent = sum(paper["year"] >= 2021 for paper in papers)
print(f"\nReviewed papers: {len(papers)} | published 2021–2026: {recent}")
assert len(papers) >= 5 and recent >= 3


year | paper                                    | approach                                   | evaluation                              | gap                                                  
-----+------------------------------------------+--------------------------------------------+-----------------------------------------+------------------------------------------------------
2022 | Alam & Hussain — Roman-Urdu-Parl         | parallel Roman-Urdu/Urdu corpus            | corpus quality and alignment            | no passage-retrieval protocol                        
2025 | Butt et al. — Roman-Urdu transliteration | transformer transliteration                | character-level transliteration quality | one conversion can remain ambiguous for search       
2025 | Butt et al. — Roman Urdu IR              | large IR dataset and multilingual baseline | ranked retrieval metrics                | translated benchmark; limited route transparency     
2024 | Wang et al. — Multilingual E5         

### Reviewed-paper references

1. M. Alam and S. U. Hussain, “Roman-Urdu-Parl: Roman-Urdu and Urdu Parallel Corpus for Urdu Language Understanding,” *ACM TALLIP*, vol. 21, no. 1, 2022. [DOI: 10.1145/3464424](https://doi.org/10.1145/3464424)
2. U. Butt, S. Varanasi, and G. Neumann, “Low-Resource Transliteration for Roman-Urdu and Urdu Using Transformer-Based Models,” *LoResMT*, 2025. [ACL Anthology](https://aclanthology.org/2025.loresmt-1.13/)
3. M. U. T. Butt, S. Varanasi, and G. Neumann, “Roman Urdu as a Low-Resource Language: Building the First IR Dataset and Baseline,” *LowResNLP*, 2025. [ACL Anthology](https://aclanthology.org/2025.lowresnlp-1.9/)
4. L. Wang *et al.*, “Multilingual E5 Text Embeddings: A Technical Report,” arXiv:2402.05672, 2024. [arXiv](https://arxiv.org/abs/2402.05672)
5. R. Nogueira and K. Cho, “Passage Re-ranking with BERT,” arXiv:1901.04085, 2019. [arXiv](https://arxiv.org/abs/1901.04085)

These same works appear in the IEEE paper bibliography so the notebook analysis and submitted paper use one consistent evidence base.


### Literature-derived research gap and proposed direction

The papers collectively provide data, transliteration, multilingual embeddings, and reranking, but they do not jointly address all four needs of this project: (1) noisy Roman entity spelling, (2) retrieval over a native Urdu-script knowledge collection, (3) CPU-first execution, and (4) an interface that exposes transformations and refuses unsupported evidence.

This gap motivates a controlled multi-route system. The initial proposal combines query views, BM25, multilingual E5, reciprocal-rank fusion, and shallow reranking. Error analysis later motivates the principal improvement: romanize Urdu article titles once and match the original Roman query using character 2–4 grams.


## Part 3 — Dataset discovery and documentation

| Dataset | Public source | Samples used | Features | Target/use | Selection reason | Known limitations |
|---|---|---:|---|---|---|---|
| Urdu Wikipedia (`wikimedia/wikipedia`, `20231101.ur`) | Hugging Face/Wikimedia snapshot | 4,000 articles; 16,352 default passages | article ID, title, URL, text, domain, passage span, token count | Gold passage ID / binary passage relevance | Native Urdu-script encyclopedic evidence with stable provenance | Only a bounded subset; encyclopedic and snapshot-limited |
| Roman-Urdu-Parl split | Public Hugging Face dataset | 30,000 supporting rows | aligned Roman-Urdu and Urdu strings | Transliteration/lexicon support only | Public script-aligned resource | Not relevance-labelled; must not be treated as retrieval ground truth |
| Project diagnostic set | Derived from the frozen corpus | 180 questions: 120 development, 60 locked test | Roman query, Urdu form, query type, domain, gold passage, exact evidence, split | Verified relevant passage ID | Enables controlled cross-script retrieval measurement | Small and title-oriented; external language review remains needed |

The primary modeling unit is a query with one verified relevant passage. Corpus passages are candidate inputs, not independent labeled training rows. The project does not claim that the auxiliary parallel corpus supplies relevance labels.


In [3]:
manifest = load_json("artifacts/metadata/phase1_manifest.json")
dataset_rows = [
    {"artifact": "Urdu Wikipedia articles", "samples": manifest["articles"]["articles"], "sha256": manifest["articles"]["sha256"][:16] + "..."},
    *[
        {"artifact": Path(item["path"]).name, "samples": item["passages"], "sha256": item["sha256"][:16] + "..."}
        for item in manifest["passage_variants"]
    ],
]
print_table(dataset_rows, ["artifact", "samples", "sha256"])


artifact                | samples | sha256             
------------------------+---------+--------------------
Urdu Wikipedia articles | 4000    | cf8abe3bc85140e7...
passages_120_24.jsonl   | 20106   | 67ddd9fdeef1f7bb...
passages_150_30.jsonl   | 16352   | 47648cf679facb9a...
passages_180_36.jsonl   | 13908   | 0fd6fbda4747e7c4...


## Provenance and integrity checks

A portable experiment must verify that the files being evaluated are the frozen files described above. The following cell recomputes passage counts and SHA-256 values rather than trusting filenames. It then checks the immutable 120/60 split and summarizes the eight query categories.


In [4]:
checks = []
for item in manifest["passage_variants"]:
    path = ROOT / item["path"]
    actual_count = sum(1 for line in path.open(encoding="utf-8") if line.strip())
    actual_digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checks.append({"file": path.name, "expected": item["passages"], "actual": actual_count, "checksum_ok": actual_digest == item["sha256"]})
print_table(checks, ["file", "expected", "actual", "checksum_ok"])

with (ROOT / "data/diagnostic/raabta_diagnostic.csv").open(encoding="utf-8-sig", newline="") as handle:
    diagnostic = list(csv.DictReader(handle))
split_counts = Counter(row["split"] for row in diagnostic)
type_counts = Counter(row["query_type"] for row in diagnostic)
print("\nSplit counts:", dict(split_counts))
print("Query-type counts:", dict(sorted(type_counts.items())))
assert all(row["expected"] == row["actual"] and row["checksum_ok"] for row in checks)
assert split_counts == {"development": 120, "test": 60}


file                  | expected | actual | checksum_ok
----------------------+----------+--------+------------
passages_120_24.jsonl | 20106    | 20106  | True       
passages_150_30.jsonl | 16352    | 16352  | True       
passages_180_36.jsonl | 13908    | 13908  | True       

Split counts: {'development': 120, 'test': 60}
Query-type counts: {'abbreviated_roman_urdu': 23, 'clean_roman_urdu': 23, 'highly_noisy_roman_urdu': 23, 'informal_spelling': 23, 'named_entity': 22, 'short_query': 22, 'slightly_ambiguous': 22, 'urdu_english_code_switching': 22}


## Part 3 conclusion

The data is public, relevant, checksum-verifiable, and separated by role. Urdu Wikipedia is the retrieval collection; Roman-Urdu-Parl is supporting script-alignment data; and the diagnostic set supplies passage-level relevance labels. The principal threats are the small title-oriented evaluation set, limited corpus coverage, one recorded gold passage per query, and the absence of independent language review. These limitations constrain every later conclusion.
